# Assignment 3 — Task 2: Diamond Price Prediction System

**Live Deployment Link:** _PASTE YOUR STREAMLIT URL HERE AFTER DEPLOYING_

**Course:** AIC354 — Machine Learning Fundamentals
**Institution:** COMSATS University Islamabad, Lahore Campus
**Instructor:** Dr. Rao Muhammad Adeel Nawab
**Student:** Mujtaba Naveed | FA24-BSE-183
**Assignment:** Assignment 3, Task 2 — Apply, Evaluate and Compare 5 ML Algorithms using 10-Fold Cross-Validation

## Step 1 — Problem Definition & Dataset

**Problem:** Predict the **price** (in USD) of a diamond from its carat weight, cut, colour, clarity, depth, table and physical dimensions (x, y, z). This is a **regression** problem, the same type as the Car Price Prediction System (predict a price from attributes).

**Dataset:** *Diamonds* (Kaggle) — https://www.kaggle.com/datasets/shivam2503/diamonds  
The **full** dataset is used (no sampling): about 54,000 diamonds, 10 input attributes plus the price.

**Evaluation:** 10-Fold Cross-Validation on the full dataset, compared using **RMSE** (Root Mean Square Error; lower is better).

**Algorithms (different from the Car Price notebooks):**
1. Ridge Regression
2. Lasso Regression
3. Extra Trees Regressor
4. AdaBoost Regressor
5. Neural Network (MLPRegressor)

## Step 2 — Import Libraries

In [ ]:
import os, sys, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, joblib
from IPython.display import display
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score

# --- the 5 algorithms ---
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("Python      :", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)

## Step 3 — Configuration
Change only this cell if you use a different dataset.

In [ ]:
CSV_NAME      = "diamond.csv"      # dataset file name
TARGET_COL    = "price"            # the column we want to predict
TARGET_UNIT   = "USD"                # unit shown in the web app
PROBLEM_TITLE = "Diamond Price Prediction"
N_FOLDS       = 10                   # 10-fold cross-validation (assignment requirement)
RANDOM_STATE  = 42

## Step 4 — Load the Full Dataset
If the file is not found, Google Colab will ask you to upload `diamond.csv`.

In [ ]:
if not os.path.exists(CSV_NAME):
    try:
        from google.colab import files
        print(f"Please upload {CSV_NAME} ...")
        uploaded = files.upload()
        CSV_NAME = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{CSV_NAME} not found. Put it in the same folder as this notebook.")

df = pd.read_csv(CSV_NAME)
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]   # drop accidental index columns
print("Columns found:", df.columns.tolist())
if TARGET_COL not in df.columns:
    matches = [c for c in df.columns if c.lower() == TARGET_COL.lower()]
    if matches:
        TARGET_COL = matches[0]
    else:
        raise KeyError(f"Column '{TARGET_COL}' not found. Set TARGET_COL in Step 3 to one of: {df.columns.tolist()}")
print(f"Loaded '{CSV_NAME}' -> {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

## Step 5 — Understand the Data

In [ ]:
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df[TARGET_COL], kde=True, ax=axes[0], color="#2a7de1")
axes[0].set_title(f"Distribution of {TARGET_COL}")

num_cols_all = df.select_dtypes(include="number")
sns.heatmap(num_cols_all.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1])
axes[1].set_title("Correlation of numeric columns")
plt.tight_layout(); plt.show()

## Step 6 — Preprocess & Encode

* Duplicate rows and rows with a missing target are removed; any missing feature values are filled (median for numbers, most frequent for text).
* **Numeric** columns are standardised (`StandardScaler`).
* **Text/categorical** columns (e.g. cut, color, clarity) are converted to numbers with one-hot encoding (`OneHotEncoder`).
* The target is also standardised internally (`TransformedTargetRegressor`) so that Lasso and the Neural Network train well; predictions are converted back to USD automatically, so RMSE is reported in real USD.

All of this is done **inside each fold** (a `Pipeline`), so no information leaks from the test fold into training.

In [ ]:
n_before = len(df)
df = df.dropna(subset=[TARGET_COL]).drop_duplicates().reset_index(drop=True)
print(f"Removed {n_before - len(df)} duplicate / unlabeled rows -> {len(df)} unique rows remain")

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(float)

numeric_cols     = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_cols = [c for c in X.columns if c not in numeric_cols]
print("Numeric columns    :", numeric_cols)
print("Categorical columns:", categorical_cols)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

def build_model(estimator):
    """preprocessing + algorithm (+ target scaling) as one object"""
    pipe = Pipeline([("prep", clone(preprocessor)), ("model", clone(estimator))])
    return TransformedTargetRegressor(regressor=pipe, transformer=StandardScaler())

## Step 7 — Select the 5 Machine Learning Algorithms

In [ ]:
ALGORITHMS = {
    "Ridge Regression":     Ridge(alpha=1.0),
    "Lasso Regression":     Lasso(alpha=0.01, max_iter=20000),
    "Extra Trees":          ExtraTreesRegressor(n_estimators=60, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1),
    "AdaBoost":             AdaBoostRegressor(n_estimators=100, random_state=RANDOM_STATE),
    "Neural Network (MLP)": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000,
                                         early_stopping=True, random_state=RANDOM_STATE),
}
print("Algorithms to compare:")
for i, name in enumerate(ALGORITHMS, 1):
    print(f"  {i}. {name}")

## Step 8 — Evaluate with 10-Fold Cross-Validation (with progress bar)
The bar shows elapsed time and the **estimated time remaining**.

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

fold_rmse = {n: [] for n in ALGORITHMS}
fold_r2   = {n: [] for n in ALGORITHMS}
fold_time = {n: [] for n in ALGORITHMS}

total_fits = len(ALGORITHMS) * N_FOLDS
with tqdm(total=total_fits, desc=f"{N_FOLDS}-fold CV", unit="fit") as bar:
    for name, estimator in ALGORITHMS.items():
        for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
            t0 = time.time()
            model = build_model(estimator)
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            pred = model.predict(X.iloc[test_idx])

            fold_rmse[name].append(np.sqrt(mean_squared_error(y.iloc[test_idx], pred)))
            fold_r2[name].append(r2_score(y.iloc[test_idx], pred))
            fold_time[name].append(time.time() - t0)

            bar.set_postfix(model=name, fold=f"{fold}/{N_FOLDS}")
            bar.update(1)
print("Cross-validation finished.")

## Step 9 — Comparison Table (RMSE)

In [ ]:
results = pd.DataFrame({
    "Algorithm":         list(ALGORITHMS),
    "Mean RMSE (USD)":   [np.mean(fold_rmse[n]) for n in ALGORITHMS],
    "Std RMSE":          [np.std(fold_rmse[n])  for n in ALGORITHMS],
    "Mean R2":           [np.mean(fold_r2[n])   for n in ALGORITHMS],
    "Mean Fit Time (s)": [np.mean(fold_time[n]) for n in ALGORITHMS],
}).sort_values("Mean RMSE (USD)").reset_index(drop=True)
results.index = results.index + 1
results.index.name = "Rank"
display(results.round(3))

print("\nRMSE of every fold:")
display(pd.DataFrame(fold_rmse, index=[f"Fold {i}" for i in range(1, N_FOLDS + 1)]).round(2))

## Step 10 — Charts

In [ ]:
order = list(results["Algorithm"])
colors = sns.color_palette("viridis", len(order))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) mean RMSE with std error bars
axes[0].bar(order, results["Mean RMSE (USD)"], yerr=results["Std RMSE"], capsize=5, color=colors)
axes[0].set_title(f"Mean RMSE ({N_FOLDS}-fold CV) — lower is better")
axes[0].set_ylabel("RMSE (USD)")
axes[0].tick_params(axis="x", rotation=30)
for i, v in enumerate(results["Mean RMSE (USD)"]):
    axes[0].text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9)

# (b) spread of RMSE across the folds
axes[1].boxplot([fold_rmse[n] for n in order])
axes[1].set_xticklabels(order)
axes[1].set_title("RMSE spread across folds")
axes[1].set_ylabel("RMSE (USD)")
axes[1].tick_params(axis="x", rotation=30)

# (c) training time
axes[2].bar(order, results["Mean Fit Time (s)"], color=colors)
axes[2].set_title("Mean training time per fold (s)")
axes[2].set_ylabel("seconds")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout(); plt.show()

## Step 11 — Conclusion

In [ ]:
best = results.iloc[0]
worst = results.iloc[-1]
best_name = best["Algorithm"]
print(f"Best algorithm : {best_name}")
print(f"  Mean RMSE    : {best['Mean RMSE (USD)']:,.2f} {TARGET_UNIT}  (std {best['Std RMSE']:,.2f})")
print(f"  Mean R2      : {best['Mean R2']:.3f}")
print(f"Worst algorithm: {worst['Algorithm']} (RMSE {worst['Mean RMSE (USD)']:,.2f} {TARGET_UNIT})")
print(f"\n{best_name} makes the smallest average error on unseen data across all {N_FOLDS} folds, "
      f"so it is selected as the final model for deployment.")

## Step 12 — Train the Final Model on the Full Dataset & Save It

The best algorithm is re-trained on **all** rows and saved as `model.pkl` (it contains the preprocessing too).  
`meta.json` stores the input fields for the web app, and `requirements.txt` pins the library versions used here, so the deployed app loads the model without version problems.

In [ ]:
final_model = build_model(ALGORITHMS[best_name])
final_model.fit(X, y)

# quick sanity check on 3 rows
sample = X.head(3)
print("Predicted:", np.round(final_model.predict(sample), 2))
print("Actual   :", y.head(3).round(2).tolist())

# ---- save model ----
joblib.dump(final_model, "model.pkl", compress=3)
size_mb = os.path.getsize("model.pkl") / (1024 * 1024)
print(f"model.pkl size: {size_mb:.1f} MB")
if size_mb > 24:
    print("WARNING: GitHub's browser upload accepts files up to 25 MB. Reduce n_estimators / raise min_samples_leaf in Step 7 and run again.")

# ---- save the description of the input form for the web app ----
features = {}
for col in X.columns:
    if col in numeric_cols:
        is_int = pd.api.types.is_integer_dtype(X[col])
        features[col] = {
            "type": "int" if is_int else "float",
            "min": float(X[col].min()), "max": float(X[col].max()), "median": float(X[col].median()),
        }
    else:
        features[col] = {"type": "category", "options": sorted(X[col].astype(str).unique().tolist())}

meta = {
    "problem_title": PROBLEM_TITLE,
    "target": TARGET_COL,
    "unit": TARGET_UNIT,
    "best_model": best_name,
    "cv_rmse": float(best["Mean RMSE (USD)"]),
    "cv_r2": float(best["Mean R2"]),
    "n_rows": int(len(df)),
    "n_folds": N_FOLDS,
    "features": features,
}
with open("meta.json", "w") as f:
    json.dump(meta, f, indent=2)

# ---- save pinned requirements for Streamlit Cloud ----
with open("requirements.txt", "w") as f:
    f.write("streamlit\n")
    f.write(f"scikit-learn=={sklearn.__version__}\n")
    f.write(f"pandas=={pd.__version__}\n")
    f.write(f"numpy=={np.__version__}\n")
    f.write(f"joblib=={joblib.__version__}\n")

print("\nSaved: model.pkl, meta.json, requirements.txt")
print(open("requirements.txt").read())

## Step 13 — Execute the Application Phase

The saved model is loaded from `model.pkl` and used to predict for one unseen record. Change the values in `user_input` to try other inputs (the web app in Step 16 does exactly this with a form).

In [ ]:
loaded_model = joblib.load("model.pkl")

# --- Take input from the user (edit these values) ---
user_input = X.iloc[0].to_dict()
print("Input:", user_input)

# --- Convert input into a feature vector (same columns, same order as training) ---
feature_vector = pd.DataFrame([user_input])[list(X.columns)]

# --- Model prediction ---
predicted = loaded_model.predict(feature_vector)[0]
print(f"\nPredicted {TARGET_COL}: {predicted:,.2f} {TARGET_UNIT}")

## Step 14 — Execute the Feedback Phase

After deployment, feedback from users and domain experts is used to improve the model. Possible improvements:
* Collect real predictions vs. actual values from users and re-train periodically.
* Tune the hyper-parameters of the best algorithm (e.g. with grid search inside cross-validation).
* Add more informative input features and more training data.
* Monitor the RMSE over time and re-train when the error grows.

## Step 15 — Download the Files
These three files go into the `streamlit-app` folder of your GitHub repo (together with `app.py`).

In [ ]:
try:
    from google.colab import files
    for name in ["model.pkl", "meta.json", "requirements.txt"]:
        files.download(name)
except ImportError:
    print("Not running on Colab — the files are in the current folder.")

## Step 16 — Deployment

The trained model is deployed on **Streamlit Community Cloud** and integrated with a web app (`app.py`) that reads the user's inputs and shows the predicted diamond price.

**Live Deployment Link:** _PASTE YOUR STREAMLIT URL HERE AFTER DEPLOYING_  
**GitHub Repository:** _PASTE YOUR REPO LINK HERE_